# 01 — Dataset integrity, vehicle IDs, and route audit

This notebook covers Tasks 1 and 3–7. It reads every raw daily CSV directly from `data/`. It writes no cleaned, filtered, or intermediate dataset.

## Cleaning/interpretation log

- All identifier and route columns are read as strings. No numeric coercion is used for IDs.
- Timestamps are parsed only for validation; source values are not overwritten.
- Exact duplicates are counted within each daily file. Since filenames partition calendar dates, cross-file duplicates would require a timestamp/date inconsistency and are separately exposed by the filename-vs-timestamp check.
- Route normalization trims surrounding whitespace and changes only an integer-like decimal suffix (`44.0 → 44`). Leading zeroes, internal punctuation, suffixes, and decimal variants such as `49.1` are preserved.
- Full-name normalization is diagnostic only: uppercase, whitespace normalization, and accent removal. It is not used to force a route match.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import re, unicodedata, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

HERE = Path.cwd().resolve()
PROJECT = HERE.parent if HERE.name == "mob_tick_matching_robustness_check" else HERE
DATA = PROJECT / "data"
MOB_FILES = sorted((DATA / "mobility_data").glob("*.csv"))
TIX_FILES = sorted((DATA / "ticket_data").glob("*.csv"))
BAD_DATES = {"2026-03-18", "2026-03-19", "2026-03-20", "2026-03-22", "2026-03-28"}
RELIABLE_FILES_M = [p for p in MOB_FILES if p.stem not in BAD_DATES]
RELIABLE_DATES = {p.stem for p in RELIABLE_FILES_M}
RELIABLE_FILES_T = [p for p in TIX_FILES if p.stem in RELIABLE_DATES]

def canon_route(x):
    """Conservative formatting normalization; does not remove leading zeroes or punctuation."""
    if pd.isna(x): return pd.NA
    s = str(x).strip()
    return re.sub(r"^(\d+)\.0$", r"\1", s)

def canon_name(x):
    if pd.isna(x): return pd.NA
    s = unicodedata.normalize("NFKD", str(x))
    s = "".join(c for c in s if not unicodedata.combining(c)).upper().strip()
    return re.sub(r"\s+", " ", s)

print(f"Project: {PROJECT}")
print(f"Mobility files: {len(MOB_FILES)}; Ticket files: {len(TIX_FILES)}; reliable overlapping dates: {len(RELIABLE_DATES)}")

Project: /Users/pirin/Desktop/Under Grad Research/Netmob2026_data_challenge
Mobility files: 19; Ticket files: 31; reliable overlapping dates: 16


In [2]:
def audit_group(files, kind):
    relevant = (["id","timestamp","tripId","lat","lng","heading","lineId","lineName","headsign","direction"] if kind=="mobility" else
                ["transaction_date","anon_user_id","fare_type","vehicle_number","company_number","route_detail_id","integration_flag","card_type","debited_amount","view_type","route_name","day_type","day_period"])
    timecol = "timestamp" if kind=="mobility" else "transaction_date"
    rows=[]; missing=Counter(); dtypes={}; values=Counter(); duplicate_total=0
    malformed_total=0; filename_date_mismatch=0
    for p in files:
        df=pd.read_csv(p, dtype={c:"string" for c in relevant}, low_memory=False)
        parsed=pd.to_datetime(df[timecol], errors="coerce", utc=(kind=="ticket"))
        malformed=int(parsed.isna().sum())
        local_date=(parsed.dt.tz_convert("America/Sao_Paulo").dt.strftime("%Y-%m-%d") if kind=="ticket" else parsed.dt.strftime("%Y-%m-%d"))
        mismatch=int((local_date.notna() & local_date.ne(p.stem)).sum())
        dup=int(df.duplicated().sum())
        duplicate_total += dup; malformed_total += malformed; filename_date_mismatch += mismatch
        for c in relevant:
            if c in df:
                missing[c] += int(df[c].isna().sum() + df[c].fillna("").str.strip().eq("").sum())
                dtypes.setdefault(c,set()).add(str(df[c].dtype))
        rows.append({"file":p.name,"rows":len(df),"timestamp_min":parsed.min(),"timestamp_max":parsed.max(),"malformed_time":malformed,"exact_duplicates":dup,"file_date_mismatch":mismatch})
        values["rows"] += len(df)
    miss=pd.DataFrame({"column":list(missing),"missing_count":list(missing.values())})
    miss["missing_rate_pct"]=100*miss.missing_count/values["rows"]
    return pd.DataFrame(rows), miss.sort_values("missing_rate_pct",ascending=False), dtypes, {"rows":values["rows"],"duplicates":duplicate_total,"malformed":malformed_total,"file_date_mismatch":filename_date_mismatch}

mob_daily,mob_missing,mob_dtypes,mob_totals=audit_group(MOB_FILES,"mobility")
tix_daily,tix_missing,tix_dtypes,tix_totals=audit_group(TIX_FILES,"ticket")
display(Markdown("## A. Dataset integrity — files, dates, row counts, ranges"))
display(mob_daily)
display(tix_daily)
display(pd.DataFrame([{"dataset":"Mobility",**mob_totals},{"dataset":"Ticket",**tix_totals}]))
display(Markdown("### Missingness of matching-relevant columns")); display(mob_missing); display(tix_missing)
display(Markdown("### Actual pandas dtypes under explicit identifier-as-string loading"))
display(pd.DataFrame({"Mobility":pd.Series({k:", ".join(v) for k,v in mob_dtypes.items()}),"Ticket":pd.Series({k:", ".join(v) for k,v in tix_dtypes.items()})}))

## A. Dataset integrity — files, dates, row counts, ranges

,file,rows,timestamp_min,timestamp_max,malformed_time,exact_duplicates,file_date_mismatch
0,2026-03-11.csv,950855,2026-03-11 00:00:07,2026-03-11 23:59:52,0,0,0
1,2026-03-12.csv,985831,2026-03-12 00:00:07,2026-03-12 23:59:52,0,0,0
2,2026-03-13.csv,1053712,2026-03-13 00:00:07,2026-03-13 23:59:52,0,0,0
3,2026-03-14.csv,774648,2026-03-14 00:00:07,2026-03-14 23:59:54,0,0,0
4,2026-03-15.csv,481931,2026-03-15 00:00:09,2026-03-15 23:59:50,0,0,0
5,2026-03-16.csv,651833,2026-03-16 00:00:09,2026-03-16 23:59:46,0,0,0
6,2026-03-17.csv,226412,2026-03-17 00:00:01,2026-03-17 10:46:51,0,0,0
7,2026-03-20.csv,420493,2026-03-20 14:52:13,2026-03-20 23:59:50,0,0,0
8,2026-03-21.csv,555022,2026-03-21 00:00:05,2026-03-21 23:58:19,0,0,0
9,2026-03-22.csv,65993,2026-03-22 00:04:06,2026-03-22 23:57:37,0,0,0


,file,rows,timestamp_min,timestamp_max,malformed_time,exact_duplicates,file_date_mismatch
0,2026-03-01.csv,63379,2026-03-01 03:00:02+00:00,2026-03-02 02:59:38+00:00,0,0,0
1,2026-03-02.csv,258057,2026-03-02 03:00:04+00:00,2026-03-03 02:59:05+00:00,0,19,0
2,2026-03-03.csv,267760,2026-03-03 03:01:12+00:00,2026-03-04 02:59:39+00:00,0,699,0
3,2026-03-04.csv,265886,2026-03-04 03:01:36+00:00,2026-03-05 02:58:44+00:00,0,21,0
4,2026-03-05.csv,269360,2026-03-05 03:01:40+00:00,2026-03-06 02:59:50+00:00,0,341,0
5,2026-03-06.csv,266077,2026-03-06 03:04:31+00:00,2026-03-07 02:58:11+00:00,0,20,0
6,2026-03-07.csv,133017,2026-03-07 03:00:03+00:00,2026-03-08 02:59:59+00:00,0,11,0
7,2026-03-08.csv,71337,2026-03-08 03:00:04+00:00,2026-03-09 02:58:27+00:00,0,2,0
8,2026-03-09.csv,242639,2026-03-09 03:03:05+00:00,2026-03-10 02:59:01+00:00,0,19,0
9,2026-03-10.csv,274380,2026-03-10 03:01:12+00:00,2026-03-11 02:59:02+00:00,0,467,0


,dataset,rows,duplicates,malformed,file_date_mismatch
0,Mobility,13850467,0,0,0
1,Ticket,6790681,7734,0,0


### Missingness of matching-relevant columns

,column,missing_count,missing_rate_pct
0,id,0,0.0
1,timestamp,0,0.0
2,tripId,0,0.0
3,lat,0,0.0
4,lng,0,0.0
5,heading,0,0.0
6,lineId,0,0.0
7,lineName,0,0.0
8,headsign,0,0.0
9,direction,0,0.0


,column,missing_count,missing_rate_pct
0,transaction_date,0,0.0
1,anon_user_id,0,0.0
2,fare_type,0,0.0
3,vehicle_number,0,0.0
4,company_number,0,0.0
5,route_detail_id,0,0.0
6,integration_flag,0,0.0
7,card_type,0,0.0
8,debited_amount,0,0.0
9,view_type,0,0.0


### Actual pandas dtypes under explicit identifier-as-string loading

,Mobility,Ticket
anon_user_id,NaN,string
card_type,NaN,string
company_number,NaN,string
day_period,NaN,string
day_type,NaN,string
debited_amount,NaN,string
direction,string,NaN
fare_type,NaN,string
heading,string,NaN
headsign,string,NaN


In [3]:
# Vehicle summaries are accumulated without retaining all raw rows.
mob_id_obs=Counter(); mob_id_days=defaultdict(set); mob_active=[]; mob_first={}; mob_last={}
tix_key_obs=Counter(); tix_key_days=defaultdict(set); tix_active=[]; vehicle_companies=defaultdict(set)
mob_raw_routes=Counter(); tix_raw_routes=Counter(); mob_names=Counter(); tix_names=Counter()
tix_route_views=defaultdict(Counter); mob_route_names=defaultdict(Counter)
tix_detail_route=defaultdict(set); tix_detail_view=defaultdict(set); tix_detail_company=defaultdict(set); route_details=defaultdict(set)

for p in MOB_FILES:
    df=pd.read_csv(p,usecols=["id","timestamp","lineId","lineName"],dtype="string")
    ids=df.id.dropna(); mob_id_obs.update(ids); active=set(ids); mob_active.append((p.stem,len(active)))
    for x in active: mob_id_days[x].add(p.stem)
    ts=pd.to_datetime(df.timestamp,errors="coerce")
    for x,g in pd.DataFrame({"id":df.id,"ts":ts}).dropna().groupby("id").ts:
        lo,hi=g.min(),g.max(); mob_first[x]=min(mob_first.get(x,lo),lo); mob_last[x]=max(mob_last.get(x,hi),hi)
    mob_raw_routes.update(df.lineId.dropna()); mob_names.update(df.lineName.dropna())
    for (r,n),cnt in df.groupby(["lineId","lineName"],dropna=True).size().items(): mob_route_names[r][n]+=int(cnt)

for p in TIX_FILES:
    df=pd.read_csv(p,usecols=["vehicle_number","company_number","route_name","view_type","route_detail_id"],dtype="string")
    for v,c in df[["vehicle_number","company_number"]].dropna().drop_duplicates().itertuples(index=False): vehicle_companies[v].add(c)
    keys=df.vehicle_number.dropna(); tix_key_obs.update(keys); active=set(keys); tix_active.append((p.stem,len(active)))
    for x in active: tix_key_days[x].add(p.stem)
    tix_raw_routes.update(df.route_name.dropna()); tix_names.update(df.view_type.dropna())
    for (r,n),cnt in df.groupby(["route_name","view_type"],dropna=True).size().items(): tix_route_views[r][n]+=int(cnt)
    for r,d,v,c in df[["route_name","route_detail_id","view_type","company_number"]].dropna(subset=["route_name","route_detail_id"]).drop_duplicates().itertuples(index=False):
        route_details[r].add(d); tix_detail_route[d].add(r); tix_detail_view[d].add(v); tix_detail_company[d].add(c)

mob_vehicle=pd.DataFrame({"id":list(mob_id_obs),"observations":[mob_id_obs[x] for x in mob_id_obs],"days_observed":[len(mob_id_days[x]) for x in mob_id_obs],"first_seen":[mob_first.get(x) for x in mob_id_obs],"last_seen":[mob_last.get(x) for x in mob_id_obs]}).sort_values("observations",ascending=False)
tix_vehicle=pd.DataFrame({"vehicle_number":list(tix_key_obs),"transactions":[tix_key_obs[x] for x in tix_key_obs],"days_observed":[len(tix_key_days[x]) for x in tix_key_obs],"company_count":[len(vehicle_companies[x]) for x in tix_key_obs],"companies":[",".join(sorted(vehicle_companies[x])) for x in tix_key_obs]}).sort_values("transactions",ascending=False)
display(Markdown("## B. Vehicle ID structure"))
display(pd.DataFrame({"metric":["unique vehicles","median days/vehicle","min days/vehicle","max days/vehicle"],"Ticket":[len(tix_vehicle),tix_vehicle.days_observed.median(),tix_vehicle.days_observed.min(),tix_vehicle.days_observed.max()],"Mobility":[len(mob_vehicle),mob_vehicle.days_observed.median(),mob_vehicle.days_observed.min(),mob_vehicle.days_observed.max()]}))
display(Markdown("### Active vehicles per day")); display(pd.DataFrame(tix_active,columns=["date","Ticket_active"]).merge(pd.DataFrame(mob_active,columns=["date","Mobility_active"]),on="date",how="outer"))
display(Markdown("### Ticket vehicle summary (complete table)")); display(tix_vehicle)
display(Markdown("### Mobility ID summary (complete table; inspect short spans as replacement clues)")); display(mob_vehicle)
multi_company=tix_vehicle.query("company_count > 1")
display(Markdown(f"### `ticket.groupby('vehicle_number')['company_number'].nunique()` result: {len(multi_company)} multi-company vehicle numbers")); display(multi_company)

## B. Vehicle ID structure

,metric,Ticket,Mobility
0,unique vehicles,556.0,527.0
1,median days/vehicle,25.0,13.0
2,min days/vehicle,1.0,1.0
3,max days/vehicle,31.0,19.0


### Active vehicles per day

,date,Ticket_active,Mobility_active
0,2026-03-01,196,NaN
1,2026-03-02,486,NaN
2,2026-03-03,487,NaN
3,2026-03-04,490,NaN
4,2026-03-05,491,NaN
5,2026-03-06,487,NaN
6,2026-03-07,289,NaN
7,2026-03-08,196,NaN
8,2026-03-09,489,NaN
9,2026-03-10,487,NaN


### Ticket vehicle summary (complete table)

,vehicle_number,transactions,days_observed,company_count,companies
76,11049,21886,31,1,3
125,24116,20478,26,1,56
204,13061,20157,28,1,2
116,11151,20064,29,1,3
92,24115,19904,26,1,56
...,...,...,...,...,...
535,21148,160,1,1,56
554,21029,157,1,1,3
423,23034,149,1,1,56
538,21006,103,1,1,3


### Mobility ID summary (complete table; inspect short spans as replacement clues)

,id,observations,days_observed,first_seen,last_seen
465,123460,55219,17,2026-03-13 13:55:22,2026-03-31 23:37:32
28,97095,52892,19,2026-03-11 00:00:07,2026-03-31 23:37:32
33,99133,51673,19,2026-03-11 00:00:07,2026-03-31 23:37:32
29,97110,50766,19,2026-03-11 00:00:07,2026-03-31 22:54:47
461,116875,50491,16,2026-03-13 13:55:22,2026-03-31 22:43:32
...,...,...,...,...,...
412,72722,900,3,2026-03-12 14:43:24,2026-03-29 18:54:19
524,83518,875,1,2026-03-26 15:43:21,2026-03-26 20:45:19
504,99160,779,1,2026-03-16 06:02:09,2026-03-16 19:14:35
525,72757,667,1,2026-03-27 17:04:59,2026-03-27 19:51:30


### `ticket.groupby('vehicle_number')['company_number'].nunique()` result: 23 multi-company vehicle numbers

,vehicle_number,transactions,days_observed,company_count,companies
40,11077,15917,28,2,"22,3"
272,11040,14491,25,2,"22,3"
382,12010,13865,24,2,"22,3"
314,11073,13764,23,2,"22,3"
298,11071,13467,26,2,"22,3"
56,11027,13441,27,2,"22,3"
218,11029,13368,25,2,"22,3"
222,11028,13354,25,2,"22,3"
209,11016,13038,27,2,"22,3"
157,11076,12194,28,2,"22,3"


In [4]:
# Exact ID overlap sanity check, preserving strings.
tset=set(tix_vehicle.vehicle_number); mset=set(mob_vehicle.id); overlap=sorted(tset&mset)
def numeric_range(s):
    x=pd.to_numeric(pd.Series(list(s)),errors="coerce")
    return {"numeric_parse_pct":100*x.notna().mean(),"min":x.min(),"max":x.max()}
display(Markdown("### Exact identifier overlap"))
display(pd.DataFrame([{"side":"Ticket vehicle_number","unique":len(tset),**numeric_range(tset)},{"side":"Mobility id","unique":len(mset),**numeric_range(mset)}]))
display(pd.DataFrame({"exact_intersection":overlap}))
print({"intersection_n":len(overlap),"pct_ticket_ids":100*len(overlap)/len(tset),"pct_mobility_ids":100*len(overlap)/len(mset)})

### Exact identifier overlap

,side,unique,numeric_parse_pct,min,max
0,Ticket vehicle_number,556,100.0,2001,24142
1,Mobility id,527,100.0,52243,136462


,exact_intersection


{'intersection_n': 0, 'pct_ticket_ids': 0.0, 'pct_mobility_ids': 0.0}


In [5]:
# Raw route formats and conservative normalization.
traw=set(tix_raw_routes); mraw=set(mob_raw_routes)
tcanon={canon_route(x) for x in traw}; mcanon={canon_route(x) for x in mraw}; inter=tcanon&mcanon
format_flags=lambda vals: pd.DataFrame({"raw":sorted(vals),"canonical":[canon_route(x) for x in sorted(vals)],"leading_zero":[bool(re.match(r"^0\d",x)) for x in sorted(vals)],"decimal":[bool(re.search(r"\.\d+$",x)) for x in sorted(vals)],"has_space":[x!=x.strip() or " " in x for x in sorted(vals)]})
display(Markdown("## C. Route compatibility — raw formats")); display(format_flags(traw)); display(format_flags(mraw))
route_summary=pd.DataFrame([{"Ticket_unique":len(tcanon),"Mobility_unique":len(mcanon),"intersection":len(inter),"Ticket_only":len(tcanon-inter),"Mobility_only":len(mcanon-inter)}])
display(route_summary); print("Ticket-only:",sorted(tcanon-inter)); print("Mobility-only:",sorted(mcanon-inter))
tix_matched=sum(n for r,n in tix_raw_routes.items() if canon_route(r) in mcanon); mob_matched=sum(n for r,n in mob_raw_routes.items() if canon_route(r) in tcanon)
print({"pct_ticket_transactions_route_present_in_mobility":100*tix_matched/sum(tix_raw_routes.values()),"pct_mobility_observations_route_present_in_ticket":100*mob_matched/sum(mob_raw_routes.values())})

def stability_table(mapping,label):
    rows=[]
    for k,c in mapping.items():
        rows.append({label:k,"distinct_names":len(c),"dominant_name":c.most_common(1)[0][0],"dominant_share":c.most_common(1)[0][1]/sum(c.values()),"all_names":" | ".join(c)})
    return pd.DataFrame(rows).sort_values(["distinct_names",label],ascending=[False,True])
tmap=stability_table(tix_route_views,"route_name"); mmap=stability_table(mob_route_names,"lineId")
display(Markdown("### Within-system route-to-name stability")); display(tmap); display(mmap)

exact_names=set(tix_names)&set(mob_names); canon_t={canon_name(x) for x in tix_names}; canon_m={canon_name(x) for x in mob_names}
print({"exact_full_name_intersection":len(exact_names),"normalized_full_name_intersection":len(canon_t&canon_m),"ticket_unique_names":len(tix_names),"mobility_unique_names":len(mob_names)})
paired=[]
for r in sorted(inter):
    tv=Counter(); mv=Counter()
    for raw,c in tix_route_views.items():
        if canon_route(raw)==r: tv.update(c)
    for raw,c in mob_route_names.items():
        if canon_route(raw)==r: mv.update(c)
    tn=tv.most_common(1)[0][0] if tv else None; mn=mv.most_common(1)[0][0] if mv else None
    paired.append({"route":r,"Ticket_view_type":tn,"Mobility_lineName":mn,"exact":tn==mn,"case_space_accent_equal":canon_name(tn)==canon_name(mn)})
display(Markdown("### Dominant full names for shared route codes (diagnostic; no fuzzy forcing)")); display(pd.DataFrame(paired))

## C. Route compatibility — raw formats

,raw,canonical,leading_zero,decimal,has_space
0,026,026,True,False,False
1,15,15,False,False,False
2,17,17,False,False,False
3,17J,17J,False,False,False
4,21,21,False,False,False
5,22,22,False,False,False
6,24.A,24.A,False,False,False
7,26A,26A,False,False,False
8,28,28,False,False,False
9,3,3,False,False,False


,raw,canonical,leading_zero,decimal,has_space
0,03,03,True,False,False
1,15,15,False,False,False
2,21,21,False,False,False
3,22,22,False,False,False
4,24,24,False,False,False
5,24.0,24,False,True,False
6,26,26,False,False,False
7,26A,26A,False,False,False
8,28,28,False,False,False
9,30,30,False,False,False


,Ticket_unique,Mobility_unique,intersection,Ticket_only,Mobility_only
0,58,51,46,12,5


Ticket-only: ['026', '17', '17J', '24.A', '3', '33F', '34', '42SL', '46P', '56', '580.M', '62B']
Mobility-only: ['03', '24', '26', '34B', '42S']
{'pct_ticket_transactions_route_present_in_mobility': 92.74738424614556, 'pct_mobility_observations_route_present_in_ticket': 96.46061753730037}


...

,route_name,distinct_names,dominant_name,dominant_share,all_names
24,42T,2,BARRETO - CENTRO (VIA RODOVIÁRIA),0.762153,BARRETO - CENTRO (VIA RODOVIARIA) | BARRETO - ...
0,026,1,FLORÁLIA,1.000000,FLORÁLIA
1,15,1,ILHA DA CONCEIÇÃO - CENTRO,1.000000,ILHA DA CONCEIÇÃO - CENTRO
2,17,1,SÃO FRANCISCO - CENTRO (VIA FROES),1.000000,SÃO FRANCISCO - CENTRO (VIA FROES)
3,17J,1,JURUJUBA VIA FROES,1.000000,JURUJUBA VIA FROES
4,21,1,FONSECA - CENTRO,1.000000,FONSECA - CENTRO
5,22,1,LARGO DO MOURA - CENTRO,1.000000,LARGO DO MOURA - CENTRO
6,24.A,1,PALMEIRAS - CENTRO,1.000000,PALMEIRAS - CENTRO
7,26A,1,MORRO DO CÉU - CENTRO,1.000000,MORRO DO CÉU - CENTRO
46,28,1,CENTRO CIRCULAR LARGO CRAVINHO VIA FONSECA,1.000000,CENTRO CIRCULAR LARGO CRAVINHO VIA FONSECA


,lineId,distinct_names,dominant_name,dominant_share,all_names
54,03,1,CENTRO X BAIRRO DE FÁTIMA,1.0,CENTRO X BAIRRO DE FÁTIMA
0,15,1,ILHA CONCEIÇÃO X CENTRO,1.0,ILHA CONCEIÇÃO X CENTRO
1,21,1,BONFIM X TERMINAL JOÃO GOULART,1.0,BONFIM X TERMINAL JOÃO GOULART
2,22,1,BONFIM X AMARAL PEIXOTO,1.0,BONFIM X AMARAL PEIXOTO
3,24,1,PALMEIRAS X TERMINAL JOÃO GOULART (VIA GRAGOATÁ),1.0,PALMEIRAS X TERMINAL JOÃO GOULART (VIA GRAGOATÁ)
55,24.0,1,PALMEIRAS X TERMINAL JOÃO GOULART (VIA GRAGOATÁ),1.0,PALMEIRAS X TERMINAL JOÃO GOULART (VIA GRAGOATÁ)
4,26,1,FLORÁRIA X CENTRO,1.0,FLORÁRIA X CENTRO
5,26A,1,MORRO CÉU X CENTRO,1.0,MORRO CÉU X CENTRO
6,28,1,LARGO DO CRAVINHO X AMARAL PEIXOTO,1.0,LARGO DO CRAVINHO X AMARAL PEIXOTO
46,30,1,Martins Torres,1.0,Martins Torres


{'exact_full_name_intersection': 0, 'normalized_full_name_intersection': 1, 'ticket_unique_names': 58, 'mobility_unique_names': 51}


### Dominant full names for shared route codes (diagnostic; no fuzzy forcing)

,route,Ticket_view_type,Mobility_lineName,exact,case_space_accent_equal
0,15,ILHA DA CONCEIÇÃO - CENTRO,ILHA CONCEIÇÃO X CENTRO,False,False
1,21,FONSECA - CENTRO,BONFIM X TERMINAL JOÃO GOULART,False,False
2,22,LARGO DO MOURA - CENTRO,BONFIM X AMARAL PEIXOTO,False,False
3,26A,MORRO DO CÉU - CENTRO,MORRO CÉU X CENTRO,False,False
4,28,CENTRO CIRCULAR LARGO CRAVINHO VIA FONSECA,LARGO DO CRAVINHO X AMARAL PEIXOTO,False,False
5,30,MARTINS TORRES,Martins Torres,False,True
6,31,PONTA DA AREIA - BELTRÃO,PONTA DA AREIA,False,False
7,32,CACHOEIRA - CENTRO,CACHOEIRA X TERMINAL RODOVIARIO,False,False
8,33,JURUJUBA - CENTRO,JURUJUBA X TERMINAL RODOVIARIO,False,False
9,34A,L.B. - CENTRO (VIA V. JARDIM),LARGO DA BATALHA X CENTRO,False,False


In [6]:
detail_rows=[]
for r,ds in route_details.items(): detail_rows.append({"route_name":r,"distinct_route_detail_id":len(ds),"route_detail_ids":" | ".join(sorted(ds))})
detail_id_rows=[]
for d in tix_detail_route:
    detail_id_rows.append({"route_detail_id":d,"n_route_name":len(tix_detail_route[d]),"n_view_type":len(tix_detail_view[d]),"n_company":len(tix_detail_company[d]),"route_names":" | ".join(sorted(tix_detail_route[d])),"companies":" | ".join(sorted(tix_detail_company[d]))})
display(Markdown("## `route_detail_id` empirical structure")); display(pd.DataFrame(detail_rows).sort_values(["distinct_route_detail_id","route_name"],ascending=[False,True])); display(pd.DataFrame(detail_id_rows).sort_values(["n_route_name","route_detail_id"],ascending=[False,True]))
display(Markdown("**Interpretation rule:** this establishes only within-Ticket relationships. Association with `tripId`, `direction`, and `headsign` is tested after high-confidence temporal pairing in Notebook 03; equality is never assumed."))

## `route_detail_id` empirical structure

,route_name,distinct_route_detail_id,route_detail_ids
48,28,2,2498 | 41
28,41BC,2,2500 | 43
35,41JB,2,2501 | 45
43,42SL,2,2505 | 385
37,42T,2,2504 | 383
15,61,2,2502 | 47
41,026,1,66
45,15,1,87
32,17,1,2731
40,17J,1,2744


,route_detail_id,n_route_name,n_view_type,n_company,route_names,companies
12,178,1,1,1,30,27
3,179,1,1,1,47,27
58,180,1,1,1,47A,27
33,181,1,1,1,47B,27
4,2461,1,1,1,67,2
51,2498,1,1,1,28,11
59,2500,1,1,1,41BC,11
60,2501,1,1,1,41JB,11
47,2502,1,1,1,61,11
37,2504,1,1,1,42T,2


**Interpretation rule:** this establishes only within-Ticket relationships. Association with `tripId`, `direction`, and `headsign` is tested after high-confidence temporal pairing in Notebook 03; equality is never assumed.

## Notebook 01 interpretation guide

Use the displayed tables to assess A–C. A Ticket key of `(company_number, vehicle_number)` is required if the multi-company table is non-empty; otherwise `vehicle_number` is empirically globally unique for this month. Abrupt Mobility first/last spans are clues, not proof, of transponder replacement. Route-code overlap percentages—not full-name similarity—determine whether route is a viable shared key.